# NB_patient_360 — Gold: Patient 360 View

Builds `adb_dev_healthcare.gold.patient_360` — **one row per patient** — by joining silver patients with hospitals, insurance providers, and aggregated lab results.

**Load Type:** Full rebuild (CREATE OR REPLACE TABLE AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE adb_dev_healthcare.gold.patient_360 AS

WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                          AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)           AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)           AS critical_test_count,
    MAX(test_date)                                                    AS last_test_date
  FROM adb_dev_healthcare.silver.lab_results
  GROUP BY patient_id
)

SELECT
  /* Key */
  p.patient_id,

  /* Demographics */
  p.first_name,
  p.last_name,
  p.date_of_birth,
  FLOOR(DATEDIFF(CURRENT_DATE(), p.date_of_birth) / 365.25)          AS age,
  p.gender,
  p.city,
  p.state,

  /* Hospital */
  p.registered_hospital_id                                           AS hospital_id,
  h.hospital_name,

  /* Insurance */
  p.insurance_provider_id,
  ip.provider_name                                                   AS insurance_provider_name,

  /* Lab summary */
  COALESCE(la.total_tests, 0)                                        AS total_tests,
  COALESCE(la.abnormal_test_count, 0)                                AS abnormal_test_count,
  COALESCE(la.critical_test_count, 0)                                AS critical_test_count,
  la.last_test_date

FROM      adb_dev_healthcare.silver.patients            p
LEFT JOIN adb_dev_healthcare.silver.hospitals            h   ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_dev_healthcare.silver.insurance_providers  ip  ON p.insurance_provider_id   = ip.insurance_provider_id
LEFT JOIN lab_agg                                        la  ON p.patient_id              = la.patient_id;

In [0]:
updated_df = spark.table("adb_dev_healthcare.gold.patient_360")
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))